In [3]:
"""Assignment 2: Regression and Classification
Dataset: Kaggle "Student Performance Factors"
"""
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

SEED = 42
CSV = "StudentPerformanceFactors.csv"
sns.set_theme(style="whitegrid")

In [4]:
# ------------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------------
def make_synthetic(n=600, seed=SEED):
    rng = np.random.default_rng(seed)
    hours = rng.normal(20, 6, n).clip(1, 40).round()
    attend = rng.normal(80, 10, n).clip(55, 100).round()
    sleep = rng.normal(7, 1.2, n).clip(4, 10).round()
    prev = rng.normal(75, 14, n).clip(50, 100).round()
    tutor = rng.integers(0, 6, n)
    activity = rng.integers(0, 6, n)
    # diminishing returns on study hours -> mild curvature
    score = (5 + 0.35 * prev + 0.20 * attend + 1.0 * hours - 0.012 * hours ** 2
             + 0.5 * sleep + 0.4 * tutor + rng.normal(0, 3.0, n))
    return pd.DataFrame({
        "Hours_Studied": hours, "Attendance": attend, "Sleep_Hours": sleep,
        "Previous_Scores": prev, "Tutoring_Sessions": tutor,
        "Physical_Activity": activity, "Exam_Score": score.clip(0, 100).round(),
    })


if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    print(f"Loaded real Kaggle file: {CSV}")
else:
    df = make_synthetic()
    print("Kaggle CSV not found -> using synthetic data with the same columns")

cols = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores",
        "Tutoring_Sessions", "Physical_Activity", "Exam_Score"]
df = df[cols].dropna().drop_duplicates().reset_index(drop=True)
print("Shape:", df.shape)
print(df.describe().round(2), "\n")

FEATURES = ["Hours_Studied", "Attendance", "Previous_Scores", "Sleep_Hours"]  # 4 predictors
X, y = df[FEATURES], df["Exam_Score"]
idx_train, idx_test = train_test_split(df.index, test_size=0.25, random_state=SEED)
Xtr, Xte, ytr, yte = X.loc[idx_train], X.loc[idx_test], y.loc[idx_train], y.loc[idx_test]


def reg_metrics(name, y_true, y_pred):
    return {"Model": name,
            "R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))}


Loaded real Kaggle file: StudentPerformanceFactors.csv
Shape: (6606, 7)
       Hours_Studied  Attendance  Sleep_Hours  Previous_Scores  \
count        6606.00     6606.00      6606.00          6606.00   
mean           19.98       79.98         7.03            75.07   
std             5.99       11.55         1.47            14.40   
min             1.00       60.00         4.00            50.00   
25%            16.00       70.00         6.00            63.00   
50%            20.00       80.00         7.00            75.00   
75%            24.00       90.00         8.00            88.00   
max            44.00      100.00        10.00           100.00   

       Tutoring_Sessions  Physical_Activity  Exam_Score  
count            6606.00            6606.00     6606.00  
mean                1.49               2.97       67.24  
std                 1.23               1.03        3.89  
min                 0.00               0.00       55.00  
25%                 1.00               2.00

In [5]:

# Q1. Multivariate Linear Regression

lin = LinearRegression().fit(Xtr, ytr)
pred_lin = lin.predict(Xte)
print("=" * 60, "\nQ1. Multivariate Linear Regression")
print("Intercept:", round(lin.intercept_, 3))
for f, c in zip(FEATURES, lin.coef_):
    print(f"  {f:<18} coef = {c:+.3f}")
res_lin = reg_metrics("Linear (degree 1)", yte, pred_lin)
print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in res_lin.items()}, "\n")

Q1. Multivariate Linear Regression
Intercept: 42.146
  Hours_Studied      coef = +0.287
  Attendance         coef = +0.199
  Previous_Scores    coef = +0.049
  Sleep_Hours        coef = -0.026
{'Model': 'Linear (degree 1)', 'R2': 0.63, 'MAE': 1.368, 'RMSE': np.float64(2.232)} 



In [6]:

# Q2. Polynomial Regression vs Linear Regression

print("=" * 60, "\nQ2. Polynomial Regression")
rows, preds_by_deg = [], {}
for d in [1, 2, 3, 4]:
    model = make_pipeline(PolynomialFeatures(d, include_bias=False),
                          StandardScaler(), LinearRegression())
    model.fit(Xtr, ytr)
    ptr, pte = model.predict(Xtr), model.predict(Xte)
    cv = cross_val_score(model, Xtr, ytr, cv=5, scoring="r2").mean()
    preds_by_deg[d] = pte
    rows.append({"Degree": d, "Train R2": r2_score(ytr, ptr), "Test R2": r2_score(yte, pte),
                 "CV R2 (5-fold)": cv, "Test MAE": mean_absolute_error(yte, pte),
                 "Test RMSE": np.sqrt(mean_squared_error(yte, pte))})
poly_table = pd.DataFrame(rows).round(4)
print(poly_table.to_string(index=False))
pred_poly = preds_by_deg[2]  # degree 2 used for the comparison below
res_poly = reg_metrics("Polynomial (degree 2)", yte, pred_poly)
print("\nLinear vs Polynomial(2) on the test set:")
print(pd.DataFrame([res_lin, res_poly]).round(3).to_string(index=False), "\n")


Q2. Polynomial Regression
 Degree  Train R2  Test R2  CV R2 (5-fold)  Test MAE  Test RMSE
      1    0.5551   0.6302          0.5556    1.3680     2.2319
      2    0.5564   0.6278          0.5549    1.3719     2.2389
      3    0.5581   0.6275          0.5491    1.3800     2.2398
      4    0.5617   0.6215          0.5397    1.3990     2.2580

Linear vs Polynomial(2) on the test set:
                Model    R2   MAE  RMSE
    Linear (degree 1) 0.630 1.368 2.232
Polynomial (degree 2) 0.628 1.372 2.239 



In [7]:

# Q3. Logistic Regression: Pass / Fail  (Pass if Exam_Score >= 65)

PASS_MARK = 65
df["Result"] = (df["Exam_Score"] >= PASS_MARK).astype(int)  # 1 = Pass, 0 = Fail
ybin_tr, ybin_te = df.loc[idx_train, "Result"], df.loc[idx_test, "Result"]
print("=" * 60, f"\nQ3. Binary Logistic Regression (Pass if score >= {PASS_MARK})")
print("Class counts:\n", df["Result"].map({1: "Pass", 0: "Fail"}).value_counts().to_string())
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logit.fit(Xtr, ybin_tr)
pred_bin = logit.predict(Xte)
print(f"\nAccuracy: {accuracy_score(ybin_te, pred_bin):.3f}")
print(classification_report(ybin_te, pred_bin, target_names=["Fail", "Pass"], digits=3))


Q3. Binary Logistic Regression (Pass if score >= 65)
Class counts:
 Result
Pass    5155
Fail    1451

Accuracy: 0.887
              precision    recall  f1-score   support

        Fail      0.794     0.673     0.729       373
        Pass      0.909     0.949     0.928      1279

    accuracy                          0.887      1652
   macro avg      0.851     0.811     0.829      1652
weighted avg      0.883     0.887     0.883      1652



In [8]:

# Q4. Multiclass Logistic Regression: Low / Medium / High

def band(s):
    return "Low" if s < 60 else ("Medium" if s < 70 else "High")

df["Performance"] = df["Exam_Score"].apply(band)
order = ["Low", "Medium", "High"]
ymc_tr, ymc_te = df.loc[idx_train, "Performance"], df.loc[idx_test, "Performance"]
print("=" * 60, "\nQ4. Multiclass Logistic Regression (Low <60, Medium 60-69, High >=70)")
print("Class counts:\n", df["Performance"].value_counts().reindex(order).to_string())
mlogit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
mlogit.fit(Xtr, ymc_tr)
pred_mc = mlogit.predict(Xte)
print(f"\nAccuracy: {accuracy_score(ymc_te, pred_mc):.3f}")
print(classification_report(ymc_te, pred_mc, labels=order, digits=3))

Q4. Multiclass Logistic Regression (Low <60, Medium 60-69, High >=70)
Class counts:
 Performance
Low         68
Medium    4913
High      1625

Accuracy: 0.875
              precision    recall  f1-score   support

         Low      0.600     0.188     0.286        16
      Medium      0.899     0.939     0.919      1247
        High      0.786     0.697     0.738       389

    accuracy                          0.875      1652
   macro avg      0.762     0.608     0.648      1652
weighted avg      0.870     0.875     0.870      1652



In [9]:

# Q5. Compare predictions with actual values

print("=" * 60, "\nQ5. Predictions vs actual (first 15 test students)")
cmp = pd.DataFrame({
    "Actual_Score": yte.values,
    "Linear_Pred": pred_lin.round(1),
    "Poly2_Pred": pred_poly.round(1),
    "Actual_Result": ybin_te.map({1: "Pass", 0: "Fail"}).values,
    "Pred_Result": pd.Series(pred_bin).map({1: "Pass", 0: "Fail"}).values,
    "Actual_Band": ymc_te.values,
    "Pred_Band": pred_mc,
})
print(cmp.head(15).to_string(index=False))
cmp.to_csv("predictions_vs_actual.csv", index=False)


Q5. Predictions vs actual (first 15 test students)
 Actual_Score  Linear_Pred  Poly2_Pred Actual_Result Pred_Result Actual_Band Pred_Band
           66         67.3        67.3          Pass        Pass      Medium    Medium
           66         63.9        64.0          Pass        Fail      Medium    Medium
           67         70.1        70.0          Pass        Pass      Medium      High
           67         68.0        67.9          Pass        Pass      Medium    Medium
           68         69.9        69.7          Pass        Pass      Medium      High
           63         61.0        61.4          Fail        Fail      Medium    Medium
           74         74.6        75.3          Pass        Pass        High      High
           65         64.3        64.4          Pass        Fail      Medium    Medium
           69         69.7        69.6          Pass        Pass      Medium    Medium
           76         72.4        72.5          Pass        Pass        High   